## Quality of Care Indicators

Compute district-year quality-of-care indicators from DHIS2 outliers-imputed routine data.

Indicators:
- testing_rate = TEST / SUSP
- treatment_rate = MALTREAT / CONF
- case_fatality_rate = MALDTH / MALADM
- prop_adm_malaria = MALADM / ALLADM
- prop_malaria_deaths = MALDTH / ALLDTH
- non_malaria_all_cause_outpatients = ALLOUT (absolute)
- presumed_cases = PRES (absolute)

Stock-out indicators are not implemented yet (on hold, NMDR data pending).

In [ ]:
# Preliminaries
options(scipen=999)

ROOT_PATH <- "~/workspace"
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
CODE_PATH <- file.path(ROOT_PATH, "code")
DATA_PATH <- file.path(ROOT_PATH, "data")
OUTPUT_DATA_PATH <- file.path(DATA_PATH, "dhis2", "quality_of_care")
FIGURES_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care", "reporting", "outputs", "figures")

dir.create(OUTPUT_DATA_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_PATH, recursive = TRUE, showWarnings = FALSE)

source(file.path(CODE_PATH, "snt_utils.r"))
required_packages <- c("jsonlite", "data.table", "arrow", "sf", "ggplot2", "glue", "reticulate", "RColorBrewer", "writexl", "dplyr")
install_and_load(required_packages)

Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
openhexa <- reticulate::import("openhexa.sdk")

config_json <- jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_config.json"))
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
DHIS2_FORMATTED_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
OUTLIERS_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_OUTLIERS_IMPUTATION

In [ ]:
# Validate data_action parameter
if (!exists("data_action")) {
  data_action <- "imputed"
}

allowed_actions <- c("imputed", "removed")
if (!(data_action %in% allowed_actions)) {
  stop(glue::glue("Invalid data_action: {data_action}. Allowed: {paste(allowed_actions, collapse=', ')}"))
}

# Automatically find the latest routine outliers-imputed file in the dataset
# Pattern: {COUNTRY_CODE}_routine_outliers-*_{data_action}.parquet
log_msg(glue::glue("Searching for latest routine outliers-imputed file in dataset (data_action: {data_action})..."))

dataset_last_version <- openhexa$workspace$get_dataset(OUTLIERS_DATASET)$latest_version
if (is.null(dataset_last_version)) {
  stop(glue::glue("[ERROR] No version available in dataset `{OUTLIERS_DATASET}`. Process stopped."))
}

# Pattern to match: {COUNTRY_CODE}_routine_outliers-*_{data_action}.parquet
pattern_prefix <- glue::glue("{COUNTRY_CODE}_routine_outliers-")
pattern_suffix <- glue::glue("_{data_action}.parquet")
routine_filename <- NULL
files_list <- reticulate::iterate(dataset_last_version$files)

# Find all matching files and select the latest one
matching_files <- c()
for (file in files_list) {
  filename <- file$filename
  if (startsWith(filename, pattern_prefix) && endsWith(filename, pattern_suffix)) {
    matching_files <- c(matching_files, filename)
  }
}

if (length(matching_files) == 0) {
  stop(glue::glue("[ERROR] No file matching pattern `{pattern_prefix}*{pattern_suffix}` found in dataset `{OUTLIERS_DATASET}`. ",
                  "Please run an outlier imputation pipeline first (e.g., snt_dhis2_outliers_imputation_mean) with `data_action=\"{data_action}\"`."))
}

# Select the latest file (alphabetically sorted, which should correspond to most recent method)
routine_filename <- sort(matching_files, decreasing = TRUE)[1]

log_msg(glue::glue("Found {length(matching_files)} matching file(s). Using latest: {routine_filename}"))

# Load the routine file
routine <- tryCatch({
  get_latest_dataset_file_in_memory(OUTLIERS_DATASET, routine_filename)
}, error = function(e) {
  msg <- paste0("[ERROR] 🛑 Error while loading DHIS2 routine data file `", routine_filename, 
                "` from `", OUTLIERS_DATASET, "`. [ERROR DETAILS] ", conditionMessage(e))
  stop(msg)
})

shapes <- get_latest_dataset_file_in_memory(DHIS2_FORMATTED_DATASET, paste0(COUNTRY_CODE, "_shapes.geojson"))

setDT(routine)

# Core required columns (must exist)
core_cols <- c("ADM2_ID", "YEAR")
core_missing <- setdiff(core_cols, names(routine))
if (length(core_missing) > 0) {
  stop(glue::glue("Missing core required columns in routine data: {paste(core_missing, collapse=', ')}"))
}

# Optional indicator columns (will be checked and handled gracefully)
indicator_cols <- c("TEST", "SUSP", "MALTREAT", "CONF", "MALDTH", "MALADM", "ALLADM", "ALLDTH", "ALLOUT", "PRES")
available_cols <- intersect(indicator_cols, names(routine))
missing_cols <- setdiff(indicator_cols, names(routine))

if (length(missing_cols) > 0) {
  log_msg(glue::glue("[WARNING] Some indicator columns are missing: {paste(missing_cols, collapse=', ')}. These indicators will not be calculated."), level = "warning")
}

# Convert available numeric columns
# Handle "-" and other non-numeric values by converting them to NA first
num_cols <- intersect(available_cols, names(routine))
if (length(num_cols) > 0) {
  for (col in num_cols) {
    # First convert to character to handle "-" strings, then replace with NA, then convert to numeric
    col_vals <- as.character(routine[[col]])
    col_vals[is.na(col_vals) | col_vals == "" | col_vals == "-"] <- NA_character_
    routine[, (col) := as.numeric(col_vals)]
  }
}
routine[, YEAR := as.integer(YEAR)]
routine[, ADM2_ID := as.character(ADM2_ID)]

# Aggregate available columns only using lapply
if (length(available_cols) > 0) {
  qoc <- routine[, lapply(.SD, function(x) sum(x, na.rm = TRUE)), 
                 .SDcols = available_cols, 
                 by = .(ADM2_ID, YEAR)]
} else {
  # If no indicator columns available, create empty structure
  qoc <- routine[, .(ADM2_ID, YEAR)]
  qoc <- unique(qoc)
}

# Calculate indicators only if required columns are available
if ("TEST" %in% names(qoc) && "SUSP" %in% names(qoc)) {
  qoc[, testing_rate := fifelse(SUSP > 0, TEST / SUSP, NA_real_)]
} else {
  log_msg("[WARNING] Cannot calculate testing_rate: missing TEST or SUSP columns", level = "warning")
}

if ("MALTREAT" %in% names(qoc) && "CONF" %in% names(qoc)) {
  qoc[, treatment_rate := fifelse(CONF > 0, MALTREAT / CONF, NA_real_)]
} else {
  log_msg("[WARNING] Cannot calculate treatment_rate: missing MALTREAT or CONF columns", level = "warning")
}

if ("MALDTH" %in% names(qoc) && "MALADM" %in% names(qoc)) {
  qoc[, case_fatality_rate := fifelse(MALADM > 0, MALDTH / MALADM, NA_real_)]
} else {
  log_msg("[WARNING] Cannot calculate case_fatality_rate: missing MALDTH or MALADM columns", level = "warning")
}

if ("MALADM" %in% names(qoc) && "ALLADM" %in% names(qoc)) {
  qoc[, prop_adm_malaria := fifelse(ALLADM > 0, MALADM / ALLADM, NA_real_)]
} else {
  log_msg("[WARNING] Cannot calculate prop_adm_malaria: missing MALADM or ALLADM columns", level = "warning")
}

if ("MALDTH" %in% names(qoc) && "ALLDTH" %in% names(qoc)) {
  qoc[, prop_malaria_deaths := fifelse(ALLDTH > 0, MALDTH / ALLDTH, NA_real_)]
  # Compatibility alias to match historical notebook export naming
  qoc[, prop_deaths_malaria := prop_malaria_deaths]
} else {
  log_msg("[WARNING] Cannot calculate prop_malaria_deaths: missing MALDTH or ALLDTH columns", level = "warning")
}

if ("ALLOUT" %in% names(qoc)) {
  qoc[, non_malaria_all_cause_outpatients := ALLOUT]
} else {
  log_msg("[WARNING] Cannot calculate non_malaria_all_cause_outpatients: missing ALLOUT column", level = "warning")
}

if ("PRES" %in% names(qoc)) {
  qoc[, presumed_cases := PRES]
} else {
  log_msg("[WARNING] Cannot calculate presumed_cases: missing PRES column", level = "warning")
}

shapes_dt <- as.data.table(sf::st_drop_geometry(shapes))
if ("ADM2_ID" %in% names(shapes_dt) && "ADM2_NAME" %in% names(shapes_dt)) {
  shapes_dt[, ADM2_ID := as.character(ADM2_ID)]
  qoc <- merge(qoc, unique(shapes_dt[, .(ADM2_ID, ADM2_NAME)]), by = "ADM2_ID", all.x = TRUE)
}

# Main district-year outputs
out_parquet <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_{data_action}.parquet"))
out_csv <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_{data_action}.csv"))
out_xlsx <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_{data_action}.xlsx"))

# Explicit district-year naming (requested format style)
out_district_parquet <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_district_year_{data_action}.parquet"))
out_district_csv <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_district_year_{data_action}.csv"))

# Build a compact year-level summary from computed indicators
summary_tbl <- unique(qoc[, .(YEAR)])
if ("testing_rate" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, qoc[, .(testing_rate = mean(testing_rate, na.rm = TRUE)), by = .(YEAR)], by = "YEAR", all.x = TRUE)
}
if ("treatment_rate" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, qoc[, .(treatment_rate = mean(treatment_rate, na.rm = TRUE)), by = .(YEAR)], by = "YEAR", all.x = TRUE)
}
if ("case_fatality_rate" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, qoc[, .(case_fatality_rate = mean(case_fatality_rate, na.rm = TRUE)), by = .(YEAR)], by = "YEAR", all.x = TRUE)
}
if ("prop_adm_malaria" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, qoc[, .(prop_adm_malaria = mean(prop_adm_malaria, na.rm = TRUE)), by = .(YEAR)], by = "YEAR", all.x = TRUE)
}
if ("prop_malaria_deaths" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, qoc[, .(prop_malaria_deaths = mean(prop_malaria_deaths, na.rm = TRUE)), by = .(YEAR)], by = "YEAR", all.x = TRUE)
}
if ("non_malaria_all_cause_outpatients" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, qoc[, .(non_malaria_all_cause_outpatients = sum(non_malaria_all_cause_outpatients, na.rm = TRUE)), by = .(YEAR)], by = "YEAR", all.x = TRUE)
}
if ("presumed_cases" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, qoc[, .(presumed_cases = sum(presumed_cases, na.rm = TRUE)), by = .(YEAR)], by = "YEAR", all.x = TRUE)
}
summary_tbl <- summary_tbl[order(YEAR)]

out_summary_parquet <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_year_summary_{data_action}.parquet"))
out_summary_csv <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_year_summary_{data_action}.csv"))

# Persist all outputs
arrow::write_parquet(qoc, out_parquet)
data.table::fwrite(qoc, out_csv)
writexl::write_xlsx(list(quality_of_care = as.data.frame(qoc)), out_xlsx)
arrow::write_parquet(qoc, out_district_parquet)
data.table::fwrite(qoc, out_district_csv)
arrow::write_parquet(summary_tbl, out_summary_parquet)
data.table::fwrite(summary_tbl, out_summary_csv)

log_msg(glue::glue("Saved outputs: {out_parquet}, {out_csv}, {out_xlsx}, {out_district_parquet}, {out_district_csv}, {out_summary_parquet}, {out_summary_csv}"))

In [ ]:
# Yearly maps by ADM2
# Ensure ADM2_ID is character in both objects (do this once before the function)
shapes$ADM2_ID <- as.character(shapes$ADM2_ID)
qoc$ADM2_ID <- as.character(qoc$ADM2_ID)

plot_yearly_map <- function(df, sf_shapes, value_col, title_prefix, filename_prefix, is_rate = TRUE) {
  # Check if value_col exists in df
  if (!(value_col %in% names(df))) {
    log_msg(glue::glue("[WARNING] Column '{value_col}' not found in data. Skipping map generation."), level = "warning")
    return(invisible(NULL))
  }
  
  # Create a local copy of sf_shapes to avoid modifying the original
  sf_shapes_local <- sf_shapes
  if (!is.character(sf_shapes_local$ADM2_ID)) {
    sf_shapes_local$ADM2_ID <- as.character(sf_shapes_local$ADM2_ID)
  }
  
  years <- sort(unique(df$YEAR))
  for (yr in years) {
    df_y <- df[YEAR == yr]
    
    # Check if df_y has any rows
    if (nrow(df_y) == 0) {
      log_msg(glue::glue("[WARNING] No data for '{value_col}' in year {yr}. Skipping map."), level = "warning")
      next
    }
    
    # Ensure ADM2_ID is character in df_y
    df_y$ADM2_ID <- as.character(df_y$ADM2_ID)
    
    # Use dplyr::left_join for sf objects to preserve geometry (use local copy)
    map_df <- dplyr::left_join(sf_shapes_local, df_y, by = "ADM2_ID")

    # Check if value_col exists in map_df after merge
    if (!(value_col %in% names(map_df))) {
      log_msg(glue::glue("[WARNING] Column '{value_col}' not found after merge for year {yr}. Skipping map."), level = "warning")
      next
    }

    vals <- map_df[[value_col]]
    finite_vals <- vals[is.finite(vals) & !is.na(vals)]
    
    # If no valid values, skip this map
    if (length(finite_vals) == 0) {
      log_msg(glue::glue("[WARNING] No valid values for '{value_col}' in year {yr}. Skipping map."), level = "warning")
      next
    }

    # Create cat column BEFORE creating the plot
    cat_vals <- NULL
    fill_palette <- NULL
    
    if (is_rate) {
      # Create cat column with proper handling of NA values
      cat_result <- tryCatch({
        cat_vals <- cut(
          vals,
          breaks = c(-Inf, 0, 0.2, 0.4, 0.6, 0.8, 1.0, Inf),
          labels = c("<0", "0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0", ">1.0"),
          include.lowest = TRUE
        )
        fill_palette <- "YlOrRd"
        TRUE  # Success
      }, error = function(e) {
        log_msg(glue::glue("[WARNING] Failed to create categories for '{value_col}' year {yr}: {conditionMessage(e)}"), level = "warning")
        FALSE  # Failure
      })
      if (!cat_result) {
        next
      }
    } else {
      cat_result <- tryCatch({
        if (length(finite_vals) > 4) {
          br <- unique(as.numeric(quantile(finite_vals, probs = seq(0, 1, 0.2), na.rm = TRUE)))
          if (length(br) < 2) {
            cat_vals <- as.factor(rep("all", nrow(map_df)))
          } else {
            cat_vals <- cut(vals, breaks = br, include.lowest = TRUE)
          }
        } else {
          cat_vals <- as.factor(vals)
        }
        fill_palette <- "Blues"
        TRUE  # Success
      }, error = function(e) {
        log_msg(glue::glue("[WARNING] Failed to create categories for '{value_col}' year {yr}: {conditionMessage(e)}"), level = "warning")
        FALSE  # Failure
      })
      if (!cat_result) {
        next
      }
    }
    
    # Check if cat_vals was created successfully
    if (is.null(cat_vals) || length(cat_vals) != nrow(map_df)) {
      log_msg(glue::glue("[WARNING] Failed to create 'cat' column for '{value_col}' in year {yr}. Skipping map."), level = "warning")
      next
    }
    
    # Check if all values are NA (cut failed) - but allow some NA values
    if (all(is.na(cat_vals))) {
      log_msg(glue::glue("[WARNING] All 'cat' values are NA for '{value_col}' in year {yr}. Skipping map."), level = "warning")
      next
    }
    
    # Add cat column using dplyr::mutate to ensure it's properly added to sf object
    map_df <- dplyr::mutate(map_df, cat = as.factor(cat_vals))
    
    # Verify cat column exists before creating plot
    if (!("cat" %in% names(map_df))) {
      log_msg(glue::glue("[WARNING] Failed to add 'cat' column to map_df for '{value_col}' in year {yr}. Skipping map."), level = "warning")
      next
    }
    
    # Create plot AFTER cat column is added
    p <- ggplot(map_df) +
      geom_sf(aes(fill = cat), color = "grey60", size = 0.1) +
      scale_fill_brewer(palette = fill_palette, na.value = "white", drop = FALSE)

    p <- p +
      theme_void() +
      labs(
        title = paste0(title_prefix, " - ", yr),
        fill = value_col,
        caption = "Source: SNT DHIS2 outliers-imputed routine data"
      ) +
      theme(
        legend.position = "bottom",
        plot.title = element_text(face = "bold", size = 12)
      )

    out_png <- file.path(FIGURES_PATH, glue::glue("{filename_prefix}_{yr}.png"))
    
    # Try to save the plot, catch any errors
    tryCatch({
      ggsave(out_png, plot = p, width = 9, height = 7, dpi = 300, bg = "white")
      log_msg(glue::glue("Saved map: {out_png}"))
    }, error = function(e) {
      log_msg(glue::glue("[WARNING] Failed to save map for '{value_col}' year {yr}: {conditionMessage(e)}"), level = "warning")
    })
  }
}

# Plot only indicators that were calculated (columns exist)
if ("testing_rate" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "testing_rate", "Testing rate (TEST / SUSP)", "testing_rate", TRUE)
}
if ("treatment_rate" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "treatment_rate", "Treatment rate (MALTREAT / CONF)", "treatment_rate", TRUE)
}
if ("case_fatality_rate" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "case_fatality_rate", "In-hospital case fatality rate (MALDTH / MALADM)", "case_fatality_rate", TRUE)
}
if ("prop_adm_malaria" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "prop_adm_malaria", "Proportion admitted for malaria (MALADM / ALLADM)", "prop_adm_malaria", TRUE)
}
if ("prop_malaria_deaths" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "prop_malaria_deaths", "Proportion of malaria deaths (MALDTH / ALLDTH)", "prop_malaria_deaths", TRUE)
}
if ("non_malaria_all_cause_outpatients" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "non_malaria_all_cause_outpatients", "Non-malaria all-cause outpatients (ALLOUT)", "allout", FALSE)
}
if ("presumed_cases" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "presumed_cases", "Presumed cases (PRES)", "presumed_cases", FALSE)
}

log_msg(glue::glue("Saved yearly maps in: {FIGURES_PATH}"))